# Notebook 5: Backtesting & Model Validation
## Basel Traffic-Light | Kupiec POF | Christoffersen | ES Validation

**Regulatory context (FRTB / Basel III):** All VaR models must be backtested daily.

| Zone   | Exceptions | Multiplier |
|--------|-----------|------------|
| Green  | 0-4       | 3.0        |
| Yellow | 5-9       | 3.4-3.8    |
| Red    | 10+       | 4.0        |

**Tests:** Kupiec POF | Christoffersen Independence | Conditional Coverage


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm

BASE_DIR    = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DATA_DIR    = os.path.join(BASE_DIR, 'data')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')

portfolio_returns = pd.read_csv(os.path.join(DATA_DIR, 'portfolio_returns.csv'), index_col=0, parse_dates=True)['portfolio_return']

CONFIDENCE_LEVEL = 0.99
ALPHA            = 1 - CONFIDENCE_LEVEL
PORTFOLIO_VALUE  = 10_000_000
print('Data loaded.')

In [ ]:
# ── 1. Compute Rolling VaR Forecasts ─────────────────────────────────────────
LOOKBACK = 250

hs_var     = portfolio_returns.rolling(LOOKBACK).quantile(ALPHA)
rolling_mu = portfolio_returns.rolling(LOOKBACK).mean()
rolling_sg = portfolio_returns.rolling(LOOKBACK).std()
param_var  = rolling_mu + norm.ppf(ALPHA) * rolling_sg

try:
    garch_df        = pd.read_csv(os.path.join(DATA_DIR, 'garch_estimates.csv'), index_col=0, parse_dates=True)
    garch_var       = garch_df['garch_var']
    GARCH_AVAILABLE = True
except Exception:
    GARCH_AVAILABLE = False
    print('GARCH estimates not found — skipping GARCH backtest.')

hs_var    = hs_var.dropna()
param_var = param_var.reindex(hs_var.index)
actual_ret = portfolio_returns.reindex(hs_var.index)

print(f'Backtest period: {hs_var.index[0].date()} to {hs_var.index[-1].date()}')
print(f'Total observations: {len(hs_var)}')

In [ ]:
# ── 2. Kupiec POF Test ────────────────────────────────────────────────────────
def kupiec_pof_test(returns, var_forecast, confidence=0.99):
    alpha      = 1 - confidence
    exceptions = (returns < var_forecast).sum()
    T          = len(returns)
    p_hat      = exceptions / T

    if p_hat == 0 or p_hat == 1:
        return {'exceptions': exceptions, 'T': T, 'p_hat': p_hat,
                'expected_exceptions': T * alpha,
                'LR_POF': np.nan, 'p_value': np.nan, 'reject_H0': False}

    LR_POF  = -2 * (exceptions * np.log(alpha/p_hat) +
                    (T - exceptions) * np.log((1-alpha)/(1-p_hat)))
    p_value = 1 - chi2.cdf(LR_POF, df=1)

    return {'exceptions': int(exceptions), 'T': T, 'p_hat': p_hat,
            'expected_exceptions': T * alpha,
            'LR_POF': LR_POF, 'p_value': p_value,
            'reject_H0': p_value < 0.05}

hs_pof    = kupiec_pof_test(actual_ret, hs_var)
param_pof = kupiec_pof_test(actual_ret, param_var)

print('=== KUPIEC POF TEST RESULTS ===')
for name, res in [('Historical Sim', hs_pof), ('Parametric', param_pof)]:
    status = 'REJECT H0' if res['reject_H0'] else 'PASS'
    print(f'\n{name}:')
    print(f'  Exceptions       : {res["exceptions"]} / {res["T"]} (expected {res["expected_exceptions"]:.1f})')
    print(f'  Exceedance rate  : {res["p_hat"]:.2%} (expected {ALPHA:.2%})')
    print(f'  LR_POF           : {res["LR_POF"]:.4f} | p-value: {res["p_value"]:.4f} | {status}')

In [ ]:
# ── 3. Christoffersen Independence Test ──────────────────────────────────────
def christoffersen_test(returns, var_forecast, confidence=0.99):
    exceptions = (returns < var_forecast).astype(int).values
    T  = len(exceptions)
    n00 = sum((exceptions[:-1] == 0) & (exceptions[1:] == 0))
    n01 = sum((exceptions[:-1] == 0) & (exceptions[1:] == 1))
    n10 = sum((exceptions[:-1] == 1) & (exceptions[1:] == 0))
    n11 = sum((exceptions[:-1] == 1) & (exceptions[1:] == 1))

    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    pi   = (n01 + n11) / T

    if pi01 <= 0 or pi11 <= 0 or pi <= 0 or pi >= 1:
        return {'LR_ind': np.nan, 'p_value': np.nan, 'reject_H0': False,
                'pi01': pi01, 'pi11': pi11, 'clustering': pi11 > pi01}

    LR_ind = -2 * (
        (n00+n01) * np.log(1-pi) + (n01+n11) * np.log(pi)
        - n00*np.log(1-pi01) - n01*np.log(pi01+1e-10)
        - n10*np.log(1-pi11+1e-10) - n11*np.log(pi11+1e-10)
    )
    p_value = 1 - chi2.cdf(LR_ind, df=1)

    return {'LR_ind': LR_ind, 'p_value': p_value, 'reject_H0': p_value < 0.05,
            'pi01': pi01, 'pi11': pi11, 'clustering': pi11 > pi01,
            'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11}

hs_ind    = christoffersen_test(actual_ret, hs_var)
param_ind = christoffersen_test(actual_ret, param_var)

print('=== CHRISTOFFERSEN INDEPENDENCE TEST ===')
for name, res in [('Historical Sim', hs_ind), ('Parametric', param_ind)]:
    status = 'REJECT (clustering!)' if res['reject_H0'] else 'PASS'
    print(f'\n{name}:')
    print(f'  pi01 (exc after no-exc): {res["pi01"]:.4f}')
    print(f'  pi11 (exc after exc)   : {res["pi11"]:.4f}  {"<-- CLUSTERING" if res["clustering"] else ""}')
    print(f'  LR_ind: {res["LR_ind"]:.4f} | p-value: {res["p_value"]:.4f} | {status}')

In [ ]:
# ── 4. Basel Traffic-Light ────────────────────────────────────────────────────
def basel_traffic_light(returns, var_forecast, window=250):
    recent_ret = returns.iloc[-window:]
    recent_var = var_forecast.iloc[-window:]
    exceptions = (recent_ret < recent_var).sum()

    if exceptions <= 4:
        zone, multiplier = 'GREEN',  3.0
    elif exceptions <= 9:
        zone, multiplier = 'YELLOW', 3.0 + (exceptions-4) * 0.08
    else:
        zone, multiplier = 'RED',    4.0

    return {'exceptions': int(exceptions), 'zone': zone, 'multiplier': multiplier}

hs_tl    = basel_traffic_light(actual_ret, hs_var)
param_tl = basel_traffic_light(actual_ret, param_var)

print('=== BASEL TRAFFIC-LIGHT (Last 250 Days) ===')
for name, res in [('Historical Sim', hs_tl), ('Parametric', param_tl)]:
    print(f'{name}: {res["zone"]} ZONE | {res["exceptions"]} exceptions | Multiplier = {res["multiplier"]:.1f}')

In [ ]:
# ── 5. Visualization ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 13))

ax = axes[0]
ax.plot(actual_ret.index, actual_ret.values, color='gray', alpha=0.5, lw=0.7, label='Portfolio Return')
ax.plot(hs_var.index,    hs_var.values,    color='blue',  lw=1.2, label='HS VaR (99%)',         alpha=0.8)
ax.plot(param_var.index, param_var.values, color='green', lw=1.2, label='Parametric VaR (99%)', alpha=0.8)
exc_hs = actual_ret[actual_ret < hs_var]
ax.scatter(exc_hs.index, exc_hs.values, color='red', s=15, zorder=5, label=f'HS Exceptions ({len(exc_hs)})')
ax.axvspan(pd.Timestamp('2008-09-01'), pd.Timestamp('2009-06-30'), alpha=0.1, color='red',    label='GFC')
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-06-30'), alpha=0.1, color='orange', label='COVID')
ax.set_title('VaR Backtest - Returns vs VaR Forecasts', fontweight='bold')
ax.legend(loc='lower left', fontsize=8, ncol=3)
ax.grid(alpha=0.3)

ax2 = axes[1]
rolling_exc = (actual_ret < hs_var).rolling(250).sum()
ax2.fill_between(rolling_exc.index, rolling_exc.values, color='steelblue', alpha=0.6, label='Rolling 250-day Exceptions')
ax2.axhline(4,  color='#2ca02c', linestyle='--', lw=1.5, label='Green threshold (4)')
ax2.axhline(9,  color='#ffbf00', linestyle='--', lw=1.5, label='Yellow threshold (9)')
ax2.axhline(10, color='#d62728', linestyle='--', lw=1.5, label='Red threshold (10)')
ax2.set_title('Rolling 250-Day Exception Count (Basel Traffic-Light)', fontweight='bold')
ax2.set_ylabel('Exception Count')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

ax3 = axes[2]
ax3.axis('off')
table_data = [
    ['Metric', 'Historical Sim', 'Parametric Normal'],
    ['Total Exceptions',    str(hs_pof['exceptions']),  str(param_pof['exceptions'])],
    ['Expected Exceptions', f"{hs_pof['expected_exceptions']:.1f}",  f"{param_pof['expected_exceptions']:.1f}"],
    ['Exceedance Rate',     f"{hs_pof['p_hat']:.2%}",   f"{param_pof['p_hat']:.2%}"],
    ['Kupiec POF p-value',  f"{hs_pof['p_value']:.4f}", f"{param_pof['p_value']:.4f}"],
    ['Kupiec Result',       'PASS' if not hs_pof['reject_H0'] else 'REJECT',
                             'PASS' if not param_pof['reject_H0'] else 'REJECT'],
    ['Independence p-val',  f"{hs_ind['p_value']:.4f}", f"{param_ind['p_value']:.4f}"],
    ['Clustering',          'Yes' if hs_ind['clustering'] else 'No',
                             'Yes' if param_ind['clustering'] else 'No'],
    ['Basel Zone (250d)',   f"{hs_tl['zone']} ({hs_tl['exceptions']} exc)",
                             f"{param_tl['zone']} ({param_tl['exceptions']} exc)"],
    ['Capital Multiplier',  f"{hs_tl['multiplier']:.1f}", f"{param_tl['multiplier']:.1f}"],
]
tbl = ax3.table(cellText=table_data[1:], colLabels=table_data[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.5)
ax3.set_title('VaR Backtesting Summary Table', fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig9_backtest_results.png'), dpi=150, bbox_inches='tight')
plt.show()

backtest_results = pd.DataFrame({
    'date':            actual_ret.index,
    'actual_return':   actual_ret.values,
    'hs_var':          hs_var.values,
    'param_var':       param_var.values,
    'hs_exception':    (actual_ret < hs_var).values,
    'param_exception': (actual_ret < param_var).values
})
backtest_results.to_csv(os.path.join(DATA_DIR, 'backtest_results.csv'), index=False)
print('\n✅ Backtest results saved.')